# 🐍 Day 7: Mini-Project - Data Pipeline

- Data pipeline using generators and itertools
- Streaming processing stages
- Filter, transform, aggregate
- Complete runnable system

## Project Overview

We'll build a data pipeline that:
- **Reads** data from a source (simulated stream)
- **Filters** records (e.g., by condition)
- **Transforms** (e.g., parse, normalize)
- **Aggregates** (e.g., count, sum)

All stages use generators for memory-efficient streaming.

## Step 1: Data Source Generator

Simulate a stream of records. In production, this could read from a file, API, or database.

In [ ]:
def read_records(records):
    """Yield records one at a time (simulates streaming)."""
    for record in records:
        yield record

data = [
    {'id': 1, 'value': 10, 'category': 'A'},
    {'id': 2, 'value': 25, 'category': 'B'},
    {'id': 3, 'value': 15, 'category': 'A'},
    {'id': 4, 'value': 30, 'category': 'B'},
    {'id': 5, 'value': 5, 'category': 'A'},
]

for r in read_records(data):
    print(r)

## Step 2: Filter Stage

Generator that yields only records matching a predicate.

In [ ]:
def filter_records(records, predicate):
    """Filter records by predicate."""
    for r in records:
        if predicate(r):
            yield r

data = [{'id': 1, 'value': 10}, {'id': 2, 'value': 25}, {'id': 3, 'value': 15}]
filtered = filter_records(data, lambda r: r['value'] >= 15)
print(list(filtered))

## Step 3: Transform Stage

Generator that applies a transform to each record.

In [ ]:
def transform_records(records, transform):
    """Apply transform to each record."""
    for r in records:
        yield transform(r)

data = [{'id': 1, 'value': 10}, {'id': 2, 'value': 20}]
transformed = transform_records(data, lambda r: {**r, 'value_doubled': r['value'] * 2})
print(list(transformed))

## Step 4: Pipeline Class

Compose stages into a pipeline. Each stage consumes from the previous.

In [ ]:
class DataPipeline:
    def __init__(self, source):
        self.source = source
        self.stages = []

    def filter(self, predicate):
        self.stages.append(('filter', predicate))
        return self

    def transform(self, func):
        self.stages.append(('transform', func))
        return self

    def run(self):
        stream = self.source
        for stage_type, arg in self.stages:
            if stage_type == 'filter':
                stream = (r for r in stream if arg(r))
            else:
                stream = (arg(r) for r in stream)
        return stream

data = [{'v': 5}, {'v': 15}, {'v': 25}]
result = DataPipeline(data).filter(lambda r: r['v'] >= 10).transform(lambda r: r['v']).run()
print(list(result))

## Step 5: Add Aggregate Stage

Final stage: reduce stream to a single value (sum, count, etc.).

In [ ]:
class DataPipeline:
    def __init__(self, source):
        self.source = source
        self.stages = []

    def filter(self, predicate):
        self.stages.append(('filter', predicate))
        return self

    def transform(self, func):
        self.stages.append(('transform', func))
        return self

    def run(self):
        stream = self.source
        for stage_type, arg in self.stages:
            if stage_type == 'filter':
                stream = (r for r in stream if arg(r))
            else:
                stream = (arg(r) for r in stream)
        return stream

    def sum(self, key=None):
        stream = self.run()
        if key:
            return sum(key(r) for r in stream)
        return sum(stream)

    def count(self):
        return sum(1 for _ in self.run())

data = [{'v': 10}, {'v': 20}, {'v': 30}]
total = DataPipeline(data).transform(lambda r: r['v']).sum()
print(f"Sum: {total}")
print(f"Count: {DataPipeline(data).count()}")

## Step 6: Full Pipeline Demo

Complete example: filter by value, transform, aggregate by category using itertools.groupby.

In [ ]:
from itertools import groupby
from operator import itemgetter

data = [
    {'id': 1, 'value': 10, 'category': 'A'},
    {'id': 2, 'value': 25, 'category': 'B'},
    {'id': 3, 'value': 15, 'category': 'A'},
    {'id': 4, 'value': 30, 'category': 'B'},
    {'id': 5, 'value': 5, 'category': 'A'},
]

class DataPipeline:
    def __init__(self, source):
        self.source = source
        self.stages = []

    def filter(self, predicate):
        self.stages.append(('filter', predicate))
        return self

    def transform(self, func):
        self.stages.append(('transform', func))
        return self

    def run(self):
        stream = self.source
        for stage_type, arg in self.stages:
            if stage_type == 'filter':
                stream = (r for r in stream if arg(r))
            else:
                stream = (arg(r) for r in stream)
        return stream

# Filter value >= 10, extract (category, value), then group and sum
filtered = DataPipeline(data).filter(lambda r: r['value'] >= 10).run()
pairs = [(r['category'], r['value']) for r in filtered]
pairs_sorted = sorted(pairs, key=itemgetter(0))
for cat, group in groupby(pairs_sorted, key=itemgetter(0)):
    total = sum(v for _, v in group)
    print(f"Category {cat}: sum = {total}")

## Step 7: Chain with itertools.chain

Process multiple sources through the same pipeline.

In [ ]:
from itertools import chain

source1 = [{'x': 1}, {'x': 2}]
source2 = [{'x': 3}, {'x': 4}]
combined = chain(source1, source2)

class DataPipeline:
    def __init__(self, source):
        self.source = source
        self.stages = []

    def transform(self, func):
        self.stages.append(('transform', func))
        return self

    def run(self):
        stream = self.source
        for _, arg in self.stages:
            stream = (arg(r) for r in stream)
        return stream

result = DataPipeline(combined).transform(lambda r: r['x'] * 2).run()
print(list(result))

## Step 8: islice for Pagination

Use itertools.islice to take first n records (e.g., pagination).

In [ ]:
from itertools import islice

def read_lines(lines):
    for line in lines:
        yield line.strip()

lines = ['a', 'b', 'c', 'd', 'e', 'f']
page1 = list(islice(read_lines(lines), 3))
print("Page 1:", page1)

# Simulate pagination: skip 3, take 3
stream = read_lines(lines)
page2 = list(islice(stream, 3, 6))
print("Page 2:", page2)

## Extensions to Try

- Add **batch** stage: group records into chunks of n
- **Parallel** processing with concurrent.futures
- **Error handling**: skip/fail on bad records
- **Metrics**: count records at each stage
- **File/API** sources: replace list with file reader or API client

## Key Takeaways

- Generators enable streaming: process one record at a time
- Pipeline = chain of generators; each stage consumes from previous
- itertools.chain combines sources; islice for pagination/limits
- groupby for aggregation (sort first)
- Fluent API (filter().transform().run()) for readable pipelines

## 🎉 Week 8 Complete!

You've completed iterators, generators, itertools, functools, builtins, context managers, and a data pipeline. Month 2, Weeks 7-8 are done. Keep building!